<a href="https://colab.research.google.com/github/ameraalraqaeh/Supermarket-Demand-Prediction-in-AI/blob/main/AMERA_grad_pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1) Install needed libraries
!pip install pandas openpyxl scikit-learn joblib

In [ ]:
# 2) Import libraries
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving supermarket_dataset_2024_200_products (1).xlsx to supermarket_dataset_2024_200_products (1).xlsx


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving supermarket_dataset_2023_200_products.xlsx to supermarket_dataset_2023_200_products.xlsx


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving supermarket_dataset_200_products.xlsx to supermarket_dataset_200_products.xlsx


In [ ]:
# 3) Put your uploaded file names here
DATA_FILES = [
    "supermarket_dataset_2023_200_products.xlsx",
    "supermarket_dataset_2024_200_products (1).xlsx",
    "supermarket_dataset_200_products.xlsx",  # 2025 file
]

MONTH_ORDER = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12,
}

In [ ]:
# 4) Load all Excel sheets and merge them into one dataset
def load_supermarket_data(files):
    frames = []

    for file_path in files:
        workbook = pd.ExcelFile(file_path)

        for sheet_name in workbook.sheet_names:
            sheet = pd.read_excel(file_path, sheet_name=sheet_name)
            sheet["Category"] = sheet_name
            frames.append(sheet)

    data = pd.concat(frames, ignore_index=True)
    return data


raw_data = load_supermarket_data(DATA_FILES)

print("Rows:", len(raw_data))
print("Products:", raw_data["ID"].nunique())
print("Years:", sorted(raw_data["Year"].unique()))
print(raw_data.head())

Rows: 23328
Products: 216
Years: [np.int64(2023), np.int64(2024), np.int64(2025)]
   ID         Product  Year     Month Week  Sales      Category
0   1  Frozen pizza 1  2023   January   W1     27  Frozen Foods
1   1  Frozen pizza 1  2023   January   W2     29  Frozen Foods
2   1  Frozen pizza 1  2023   January   W3     29  Frozen Foods
3   1  Frozen pizza 1  2023  February   W1     26  Frozen Foods
4   1  Frozen pizza 1  2023  February   W2     30  Frozen Foods


In [ ]:
# 5) Prepare features for training
def prepare_features(data):
    data = data.copy()

    data["Month_Num"] = data["Month"].map(MONTH_ORDER)
    data["Week_Num"] = data["Week"].astype(str).str.replace("W", "", regex=False).astype(int)

    data["Time_Index"] = (data["Year"] - data["Year"].min()) * 48
    data["Time_Index"] += (data["Month_Num"] - 1) * 4 + data["Week_Num"]

    data = data.sort_values(["ID", "Time_Index"]).reset_index(drop=True)

    product_group = data.groupby("ID")["Sales"]

    data["Sales_Lag_1"] = product_group.shift(1)
    data["Sales_Lag_4"] = product_group.shift(4)

    data["Sales_Rolling_4"] = (
        product_group
        .shift(1)
        .rolling(4)
        .mean()
        .reset_index(level=0, drop=True)
    )

    data["Sales_Rolling_12"] = (
        product_group
        .shift(1)
        .rolling(12)
        .mean()
        .reset_index(level=0, drop=True)
    )

    # Target: next week sales
    data["Next_Week_Sales"] = product_group.shift(-1)

    data = data.dropna().reset_index(drop=True)
    return data


data = prepare_features(raw_data)

print(data.head())
print("Prepared rows:", len(data))

   ID         Product  Year Month Week  Sales      Category  Month_Num  \
0   1  Frozen pizza 1  2023   May   W1     18  Frozen Foods          5   
1   1  Frozen pizza 1  2023   May   W2     36  Frozen Foods          5   
2   1  Frozen pizza 1  2023   May   W3     14  Frozen Foods          5   
3   1  Frozen pizza 1  2023  June   W1     31  Frozen Foods          6   
4   1  Frozen pizza 1  2023  June   W2     19  Frozen Foods          6   

   Week_Num  Time_Index  Sales_Lag_1  Sales_Lag_4  Sales_Rolling_4  \
0         1          17         13.0         17.0            20.75   
1         2          18         18.0         25.0            21.00   
2         3          19         36.0         28.0            23.75   
3         1          21         14.0         13.0            20.25   
4         2          22         31.0         18.0            24.75   

   Sales_Rolling_12  Next_Week_Sales  
0         25.166667             36.0  
1         24.416667             14.0  
2         25.0000

In [ ]:
# 6) Split data: train on 2023 + 2024, test on 2025
FEATURES = [
    "ID",
    "Product",
    "Category",
    "Year",
    "Month_Num",
    "Week_Num",
    "Time_Index",
    "Sales",
    "Sales_Lag_1",
    "Sales_Lag_4",
    "Sales_Rolling_4",
    "Sales_Rolling_12",
]

TARGET = "Next_Week_Sales"

train_data = data[data["Year"] < 2025]
test_data = data[data["Year"] == 2025]

X_train = train_data[FEATURES]
y_train = train_data[TARGET]

X_test = test_data[FEATURES]
y_test = test_data[TARGET]

print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))

Training rows: 12960
Testing rows: 7560


In [ ]:
# 7) Build and train AI model
categorical_features = ["Product", "Category"]
numeric_features = [col for col in FEATURES if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=250,
                random_state=42,
                min_samples_leaf=2,
                n_jobs=-1,
            ),
        ),
    ]
)

model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [ ]:
# 8) Evaluate model
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("Mean Absolute Error:", round(mae, 2))
print("R2 Score:", round(r2, 3))

Mean Absolute Error: 3.91
R2 Score: 0.924


In [ ]:
# 9) Create prediction table
prediction_table = test_data[
    ["ID", "Product", "Category", "Year", "Month", "Week", "Sales", "Next_Week_Sales"]
].copy()

prediction_table["Predicted_Next_Week_Sales"] = predictions.round(2)

prediction_table["Absolute_Error"] = (
    prediction_table["Next_Week_Sales"] -
    prediction_table["Predicted_Next_Week_Sales"]
).abs().round(2)

prediction_table.head(20)

,ID,Product,Category,Year,Month,Week,Sales,Next_Week_Sales,Predicted_Next_Week_Sales,Absolute_Error
60,1,Frozen pizza 1,Frozen Foods,2025,January,W1,27,24.0,29.12,5.12
61,1,Frozen pizza 1,Frozen Foods,2025,January,W2,24,29.0,29.19,0.19
62,1,Frozen pizza 1,Frozen Foods,2025,January,W3,29,25.0,29.63,4.63
63,1,Frozen pizza 1,Frozen Foods,2025,February,W1,25,31.0,29.66,1.34
64,1,Frozen pizza 1,Frozen Foods,2025,February,W2,31,28.0,29.96,1.96
65,1,Frozen pizza 1,Frozen Foods,2025,February,W3,28,32.0,29.80,2.20
66,1,Frozen pizza 1,Frozen Foods,2025,March,W1,32,32.0,29.33,2.67
67,1,Frozen pizza 1,Frozen Foods,2025,March,W2,32,24.0,28.71,4.71
68,1,Frozen pizza 1,Frozen Foods,2025,March,W3,24,23.0,28.92,5.92
69,1,Frozen pizza 1,Frozen Foods,2025,April,W1,23,22.0,29.23,7.23


In [ ]:
# 10) Create reorder alerts
# Example values:
# current_stock = current quantity available in inventory
# safety_stock = minimum allowed stock before reorder alert

CURRENT_STOCK = 120
SAFETY_STOCK = 30

prediction_table["Example_Current_Stock"] = CURRENT_STOCK

prediction_table["Expected_Stock_After_Next_Week"] = (
    prediction_table["Example_Current_Stock"] -
    prediction_table["Predicted_Next_Week_Sales"]
).round(2)

prediction_table["Needs_Reorder"] = (
    prediction_table["Expected_Stock_After_Next_Week"] <= SAFETY_STOCK
)

prediction_table[
    [
        "Product",
        "Category",
        "Month",
        "Week",
        "Predicted_Next_Week_Sales",
        "Expected_Stock_After_Next_Week",
        "Needs_Reorder",
    ]
].head(20)

,Product,Category,Month,Week,Predicted_Next_Week_Sales,Expected_Stock_After_Next_Week,Needs_Reorder
60,Frozen pizza 1,Frozen Foods,January,W1,29.12,90.88,False
61,Frozen pizza 1,Frozen Foods,January,W2,29.19,90.81,False
62,Frozen pizza 1,Frozen Foods,January,W3,29.63,90.37,False
63,Frozen pizza 1,Frozen Foods,February,W1,29.66,90.34,False
64,Frozen pizza 1,Frozen Foods,February,W2,29.96,90.04,False
65,Frozen pizza 1,Frozen Foods,February,W3,29.80,90.20,False
66,Frozen pizza 1,Frozen Foods,March,W1,29.33,90.67,False
67,Frozen pizza 1,Frozen Foods,March,W2,28.71,91.29,False
68,Frozen pizza 1,Frozen Foods,March,W3,28.92,91.08,False
69,Frozen pizza 1,Frozen Foods,April,W1,29.23,90.77,False


In [ ]:
# 11) Show only products that need reorder
reorder_alerts = prediction_table[prediction_table["Needs_Reorder"] == True]

reorder_alerts[
    [
        "Product",
        "Category",
        "Month",
        "Week",
        "Predicted_Next_Week_Sales",
        "Expected_Stock_After_Next_Week",
        "Needs_Reorder",
    ]
].head(30)

,Product,Category,Month,Week,Predicted_Next_Week_Sales,Expected_Stock_After_Next_Week,Needs_Reorder
2355,Ice cream 1,Frozen Foods,June,W1,113.70,6.30,True
2356,Ice cream 1,Frozen Foods,June,W2,115.50,4.50,True
2357,Ice cream 1,Frozen Foods,June,W3,115.10,4.90,True
2358,Ice cream 1,Frozen Foods,July,W1,114.54,5.46,True
2359,Ice cream 1,Frozen Foods,July,W2,116.45,3.55,True
2360,Ice cream 1,Frozen Foods,July,W3,113.21,6.79,True
2361,Ice cream 1,Frozen Foods,August,W1,115.76,4.24,True
2362,Ice cream 1,Frozen Foods,August,W2,115.23,4.77,True
2363,Ice cream 1,Frozen Foods,August,W3,96.45,23.55,True
2450,Ice cream 2,Frozen Foods,June,W1,109.10,10.90,True


In [ ]:
# 12) Save model and predictions
joblib.dump(model, "inventory_sales_forecast_model.pkl")
prediction_table.to_csv("weekly_sales_predictions.csv", index=False)

print("Saved model: inventory_sales_forecast_model.pkl")
print("Saved predictions: weekly_sales_predictions.csv")

Saved model: inventory_sales_forecast_model.pkl
Saved predictions: weekly_sales_predictions.csv
